In [ ]:
%pip install xlrd

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the dataset
superstore_data = pd.read_excel(r'c:\Users\chery\Documents\di194\Week3\Day5\US Superstore data.xls')

# Check the first few rows and data types
superstore_data.info()

In [ ]:
superstore_data.head()

In [ ]:
# Checking for null values
superstore_data.isnull().sum()

In [ ]:
# Convert postal code to a string to avoid calculations on the value
superstore_data['Postal Code'] = superstore_data['Postal Code'].astype(str)

# Verify the change
superstore_data['Postal Code'].dtype

In [ ]:
# Check for duplicate rows
duplicate_count = superstore_data.duplicated().sum()
print(f"Number of duplicated rows: {duplicate_count}")

In [ ]:
#Calculate shipping time
superstore_data['Ship Time'] = (superstore_data['Ship Date'] - superstore_data['Order Date']).dt.days

In [ ]:
# Review categorial columns to understand number of unique values in each
print("Unique Counts:")
cols = ['Segment', 'Country', 'Region', 'Category', 'Sub-Category']
print(superstore_data[cols].nunique())

print("\nSegments:", superstore_data['Segment'].unique())
print("Regions:",superstore_data['Region'].unique())
print("Categories:", superstore_data['Category'].unique())

In [ ]:
# Create a profit margin column
superstore_data['Profit Margin %'] = (superstore_data['Profit'] / superstore_data['Sales']) * 100

superstore_data[['Sales', 'Profit', 'Profit Margin %']].head()

In [ ]:
# Which states have the most sales?

# Group by state and sum sales per state
state_sales = superstore_data.groupby('State')['Sales'].sum().reset_index()
state_sales = state_sales.sort_values(by='Sales', ascending=False)

state_sales.head(10)

In [ ]:
# Visualize top sales by state
sns.set_theme(style='whitegrid')

plt.Figure(figsize=(12,8))
sns.barplot(data=state_sales.head(10), x='Sales', y='State', hue='State', palette='viridis', legend=False)

plt.title('Top 10 States by Total Sales', fontsize=15)
plt.xlabel('Total Sales ($)', fontsize=12)
plt.ylabel('State', fontsize=12)

plt.show()


In [ ]:
# Look at both sales and profit
state_metrics = superstore_data.groupby('State')[['Sales', 'Profit']].sum().reset_index()
state_metrics = state_metrics.sort_values(by='Sales', ascending=False).head(10)

# Create two subplots
fig, axes = plt.subplots(1,2, figsize=(16,8), sharey=True)

# Plot the first - Total Sales (Left side)
sns.barplot(ax=axes[0], data=state_metrics, x='Sales', y='State', hue='State', palette='viridis', legend=False)
axes[0].set_title('Top 10 States by Sales', fontsize=14)
#axes[0].set_xlabel('Total Sales ($)')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.0f', padding=3)

# Plot the second - Total Profit (Right side)
sns.barplot(ax=axes[1], data=state_metrics, x='Profit', y='State', hue='State', palette='magma', legend=False)
axes[1].set_title('Profit for the Top 10 States by Sales', fontsize=14)
#axes[1].set_xlabel('Total Profit ($)')
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.0f', padding=3)

# Add a vertical line at 0 for the profit chart to highlight losses
axes[1].axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

Question: What is the difference between New York and California in terms of sales and profit? (Compare the total sales and profit between New York and California.)?

California has the highest sales at $457,688, with New York following at $310,876 (a difference of 47.2%). 
California also had the highest profit at $76,381, with New York folowing very close behind at $74,039 (a difference of only 3.1%). This implies that while California sales are much higher, the profit margin in New York is much greater, and therefore New York is a more worthwhile market to invest in. We can also see this by adding in Profit Margin to the comparison

In [ ]:
state_metrics = superstore_data.groupby('State')[['Sales', 'Profit']].sum().reset_index()
state_metrics['Profit Margin %'] = (state_metrics)['Profit'] / state_metrics['Sales'] * 100

state_metrics = state_metrics.sort_values(by='Sales', ascending=False).head(10)

fig, axes = plt.subplots(1,3, figsize=(22,8), sharey=True)

# Plot Sales
sns.barplot(ax=axes[0], data=state_metrics, x='Sales', y='State', hue='State', palette='viridis', legend=False)
axes[0].set_title('Top 10 States by Sales')
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.0f', padding=3)

# Plot Profit
sns.barplot(ax=axes[1], data=state_metrics, x='Profit', y='State', hue='State', palette='magma', legend=False)
axes[1].set_title('Total Profit ($)')
axes[1].axvline(0, color='red', linestyle='--')
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.0f', padding=3)

# Plot Profit Margin
sns.barplot(ax=axes[2], data=state_metrics, x='Profit Margin %', y='State', hue='State', palette='coolwarm', legend=False)
axes[2].set_title('Profit Margin (%)')
axes[2].axvline(0, color='red', linestyle='--')
for c in axes[2].containers: axes[2].bar_label(c, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

Question: Are there any differences among states in profitability?

While California his high sales and high profit, the high profit is due to sheer volume but the profit margin is only 16.7%, lower than the profit margins of other states, including Michigan, Virginia, Washington, and New York. New York sits at the sweet spot of having high sales with a high profit margin as well. Michigan, Virginia and Washington have lower total sales volume, however the profit margins are significant making them very profitable states and worthwhile investments. Looking at Texas, Pennsylvania, Illinois, and Ohio, they all have high sales volumes (in the top 10), but they are negative profitability. 

Question: Who is an outstanding customer in New York?

In [ ]:
# Create a new dataset with only New York data
ny_data = superstore_data[superstore_data['State'] == 'New York']

# Find the top customer by total profit
top_customer_profit = ny_data.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False).head(1)

# Find the top customer by total sales
top_customer_sales = ny_data.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(1)

print(f"Outstanding customer by Profit: \n{top_customer_profit}")
print(f"\nOutstanding customer by Sales: \n{top_customer_sales}")

In [ ]:
# Filter for New York and calculate the average profit per customer in NY
ny_data = superstore_data[superstore_data['State'] == 'New York']
ny_customer_profit = ny_data.groupby('Customer Name')['Profit'].sum().reset_index()
avg_ny_profit = ny_customer_profit['Profit'].mean()

# Get the top 5 customers by profit
top_5_ny = ny_customer_profit.sort_values(by='Profit', ascending=False).head(5)

# plot
plt.figure(figsize=(12,6))
sns.barplot(data=top_5_ny, x='Profit', y='Customer Name', hue='Customer Name', palette='Blues_r', legend=False)
plt.axvline(avg_ny_profit, color='red', linestyle='--', label=f"NY Avg Profit (${avg_ny_profit: .2f})")

plt.title('Top 5 Outstanding Customers in New York by Profit', fontsize=15)
plt.xlabel('Total Profit ($)')
plt.legend()

for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt='$%.0f', padding=3)

plt.show()

Question:
The Pareto Principle, also known as the 80/20 rule, is a concept derived from the work of Italian economist Vilfredo Pareto. It states that roughly 80% of the effects come from 20% of the causes. For instance, identifying the top 20% of products that generate 80% of sales or the top 20% of customers that contribute to 80% of profit can help in prioritizing efforts and resources. This focus can lead to improved efficiency and effectiveness in business strategies. Can we apply Pareto principle to customers and Profit ? (Determine if 20% of the customers contribute to 80% of the profit.)

In [ ]:
# Group by customer and sort by descending profit to see the customers with the highest profit
customer_analysis = superstore_data.groupby('Customer ID')['Profit'].sum().sort_values(ascending=False).reset_index()

# Calculate cumulative profit % 
total_profit = customer_analysis['Profit'].sum()
customer_analysis['Cumulative_Profit_Pct'] = (customer_analysis['Profit'].cumsum() / total_profit) * 100

# Calculate customer percentage (rank / total customers)
num_customers = len(customer_analysis)
customer_analysis['Customer_Pct'] = (customer_analysis.index + 1) / num_customers * 100

# Find the profit % contributed by exactly the top 20% of customers
top_20_threshold = customer_analysis[customer_analysis['Customer_Pct'] <= 20].iloc[-1]

print(f"Top 20% of customers contribute {top_20_threshold['Cumulative_Profit_Pct']:.2f}% of total profit.")

Question: 
* What are the Top 20 cities by Sales?
* What about the Top 20 cities by Profit?
* Are there any difference among cities in profitability?
* (Identify the top 20 cities based on total sales and total profit and analyze differences in profitability among these cities.)

In [ ]:
# Aggregate metrics by city
city_metrics = superstore_data.groupby('City')[['Sales', 'Profit']].sum().reset_index()
city_metrics['Profit Margin %'] = (city_metrics['Profit'] / city_metrics['Sales']) * 100

# select the top 20 cities by sales
top_20_cities = city_metrics.sort_values(by='Sales', ascending=False).head(20)

# create a dual subplot visualization
fig, axes = plt.subplots(1, 2, figsize=(20,10), sharey=True)

# Plot 1
sns.barplot(ax=axes[0], data=top_20_cities, x='Sales', y='City', hue='City', palette='viridis', legend=False)
axes[0].set_title('Top 20 Cities by Total Sales', fontsize=16)
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.0f', padding=3)

# Plot 2 
sns.barplot(ax=axes[1], data=top_20_cities, x='Profit Margin %', y='City', hue='City', palette='coolwarm', legend=False)
axes[1].set_title('Profit Margin % for Top Sales Cities', fontsize=16)
axes[1].axvline(0, color='red', linestyle='--')
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 based on Profit
top_20_profit = city_metrics.sort_values(by='Profit', ascending=False).head(20)

# create the figure
fig, axes = plt.subplots(1,2, figsize=(20,10), sharey=True)

# Plot 1 
sns.barplot(ax=axes[0], data=top_20_profit, x='Profit', y='City', hue='City', palette='magma', legend=False)
axes[0].set_title('Top 20 Cities by Total Profit', fontsize=15)
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.0f', padding=3)

# Plot 2 
sns.barplot(ax=axes[1], data=top_20_profit, x='Profit Margin %', y='City', hue='City', palette='viridis', legend=False)
axes[1].set_title('Profit Margin % for these Top Cities (by Profit)', fontsize=15)
axes[1].axvline(0, color='red', linestyle='--')
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

Question: What are the Top 20 customers by Sales?

In [ ]:
# Aggregate by Customer Name
customer_metrics = superstore_data.groupby('Customer Name')[['Sales', 'Profit']].sum().reset_index()
customer_metrics['Customer Profit Margin %'] = (customer_metrics['Profit'] / customer_metrics['Sales']) * 100

# get the top 20 customers based on sales
top_20_cust_sales = customer_metrics.sort_values(by='Sales', ascending=False).head(20)

# Create visualization
fig, axes = plt.subplots(1,3, figsize=(24,12), sharey=True)

# Plot 1: Sales Volume
sns.barplot(ax=axes[0], data=top_20_cust_sales, x='Sales', y='Customer Name', hue='Customer Name', palette='viridis', legend=False)
axes[0].set_title('Top 20 Customers by Total Sales', fontsize=15)
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.0f', padding=3)

# plot 2: Profit 
sns.barplot(ax=axes[1], data=top_20_cust_sales, x='Profit', y='Customer Name', hue='Customer Name', palette='magma', legend=False)
axes[1].set_title('Profit Contribution', fontsize=15)
axes[1].axvline(0, color='red', linestyle='--')
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.0f', padding=3)

# Plot 3: Profit Margin % (Efficiency)
sns.barplot(ax=axes[2], data=top_20_cust_sales, x='Customer Profit Margin %', y='Customer Name', hue='Customer Name', palette='coolwarm', legend=False)
axes[2].set_title('Customer Profit Margin %', fontsize=15)
axes[2].axvline(0, color='red', linestyle='--')
for  c in axes[2].containers: axes[2].bar_label(c, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

Question: Plot the Cumulative curve in Sales by Customers. Can we apply Pareto principle to customers and Sales ?

In [ ]:
sales_pareto = superstore_data.groupby('Customer ID')['Sales'].sum().sort_values(ascending=False).reset_index()
sales_pareto['Cumulative_Sales_Pct'] = (sales_pareto['Sales'].cumsum() / sales_pareto['Sales'].sum()) * 100
sales_pareto['Customer_Pct'] = (np.arange(len(sales_pareto)) + 1) / len(sales_pareto) * 100

plt.figure(figsize=(10,6))
plt.plot(sales_pareto['Customer_Pct'], sales_pareto['Cumulative_Sales_Pct'], color='blue', linewidth=2)

plt.axhline(80, color='red', linestyle='--', label='80% Sales Threshold')
plt.axvline(20, color='green', linestyle='--', label='20% Customer Threshold')

actual_val_at_20 = sales_pareto[sales_pareto['Customer_Pct'] <=20].iloc[-1]['Cumulative_Sales_Pct']

plt.title('Pareto Analysis: Cumulative Sales by Customer %', fontsize=15)
plt.xlabel('% of Total Customers', fontsize=12)
plt.ylabel('Cumulative % of Total Sales', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

print(f"Result: The top 20% of customers contribute {actual_val_at_20:.2f}% of total sales.")
plt.show()

It looks as though the top 20% of total customers makes up for about 50% of total sales. The pareto principle does not apply to customer and sales. However, as observed earlier, it does apply to customer and profit. 

Question: Based on the analysis, make decisions on which states and cities to prioritize for marketing strategies.

Cities Lafayette, Atlanta, and Minneapolis each have a higher than 40% profit margin, though they still make up only a small amount of total profit. They are untapped markets 